1. 物理隔离与权限锁死（最关键一步）
永远不要让智能体直接在原始数据目录下工作。

原始数据只读化：下载完 FASTQ 文件后，立刻撤销写权限。即使智能体发疯，也删不掉源文件。

Bash
chmod -R 444 /path/to/your/raw_data/
工作目录软链接：在你的项目专属目录（例如 /home/gao/projects/2026_Item13_biomini）下建立分析文件夹，把原始数据软链接过来，让智能体只在这个受限的沙盒里折腾。

2. “谋定而后动”范式（Plan -> Review -> Execute）
严禁让智能体以“Auto-run（自动执行）”模式全权接管终端。

智能体的角色是“代码生成器与规划师”，而不是“盲目执行者”。

要求智能体输出完整的 .sh 脚本或 Nextflow (.nf) 配置文件，而不是零散的命令。

新手在 VS Code 或 Vim 中打开生成的脚本，利用阿里灵码或通义千问审查一遍代码逻辑，确认没有高危命令后再手动提交到后台运行（配合 tmux）。

二、 严谨与复现：如何固定参数、版本与结果？
生信分析最忌讳“黑盒”。如果智能体每次跑出来结果都不一样，那这不叫科研，叫抽卡。

1. 彻底锁定运行环境（版本固定）
不要让智能体自己去猜或者临时安装软件，它极容易把底层依赖搞崩。

提前用 mamba 配置好专属环境（比如你现有的 regular_bioinfo 或 DE_R45）。

在给智能体的 Prompt 中硬性规定环境。例如：“所有分析必须基于我已有的 regular_bioinfo 环境中的工具（如 fastp 0.23, STAR 2.7），严禁在脚本中包含任何 conda install 或 apt-get 命令。”

分析完成后，顺手导出一份环境快照存档：

Bash
mamba env export > environment_biomni_run.yml

2. 剥夺智能体的“参数自由裁量权”
新手不知道参数怎么设，智能体往往会给一套“默认万金油”参数。

分步逼问法：不要直接对智能体说“帮我比对数据”。正确的 Prompt 应该是：“我要对哺乳动物的 RNA-seq 数据进行比对，请列出 STAR 软件影响比对率的 3 个最核心参数，并解释在有参考基因组的情况下，这 3 个参数的最佳设置建议及依据。”

确定参数后，要求它将所有参数全部显式写在脚本头部作为变量（不要隐藏在冗长的命令中），方便随时核对和复现。

三、 摆脱依赖：如何利用老手的“少数示范”建立严谨框架？
新手要做到严谨，最好的方法是“Few-Shot Prompting（少样本提示）”，即向智能体喂食老手的黄金模板。

1. 建立“SOP 知识库”
向老手要一份他们过去做类似分析的经典脚本（哪怕只是一个 Bash 脚本或者 Markdown 记录），这个文件就是你的“护身符”。

2. 将老手经验转化为智能体的“系统指令 (System Prompt)”
在使用 BioMNI 或通用大模型时，先扔给它这段话和模板：

“你现在是一个严谨的生物信息学专家。我提供了一份资深研究员编写的 RNA-seq 质控与比对标准流程（见下文代码）。请你仔细学习该模板的代码风格、参数设定逻辑以及文件输出规范。
现在，我有一批新的双端测序数据。请完全模仿上述模板的结构和严谨度，为我的新项目生成一套完整的分析脚本。如果遇到模板中未涵盖的新情况，请给出至少两种解决方案并解释利弊，等我确认后再写代码。”

通过这种方式，智能体的输出就会被强行约束在“老手经验”的轨道上，极大降低了新手犯低级常识错误的概率。

四、 智能体应用规范总结表

| 核心问题 | 错误做法（灾难根源） | 智能体严谨工作流规范 |
| :--- | :--- | :--- |
| **服务器安全** | 直接给智能体 root 或无限制操作权限。 | 仅在 `/home/.../项目目录` 中运行，原始数据设置 `chmod 444` 只读。 |
| **工作流控制** | 让智能体自动执行每一步，终端直接打印结果。 | 智能体仅生成完整 `.sh` 脚本，在 VS Code/Vim 审查后再放进 `tmux` 运行。 |
| **环境与版本** | 允许智能体自由调用 `pip install` 临时装软件。 | 强制声明必须使用现有的 `mamba activate regular_bioinfo` 等固定环境。 |
| **参数确定** | 对参数不闻不问，直接使用智能体的生成代码。 | 要求智能体详细列出核心参数的设定逻辑，并将参数显式提炼为脚本头部变量。 |
| **复用老手经验** | 每次从零开始让智能体发散思考。 | 把老手的历史脚本作为 Context 喂给大模型，要求其进行“格式与逻辑模仿”。 |

1. 软链接能防御什么？（防误删与省空间）
在生信分析中，最大的灾难通常不是“改写”了 FASTQ 测序数据（因为分析工具一般只读取它），而是 AI 或新手一不小心执行了删除命令。

如果发生 rm -rf 灾难：
假设 AI 智能体在你的工作目录里发疯，执行了 rm -rf *。
因为当前目录里只有软链接，Linux 只会把这个“快捷方式”删掉。远在 /path/to/raw_data/ 里的原始测序文件毫发无损。

极度节省空间：
动辄几十上百 GB 的 RNA-seq 数据，如果每次分析都复制（cp）一份到工作目录，服务器那 20TB 的硬盘也很快会撑爆。软链接几乎不占用任何物理磁盘空间。

2. 软链接防不住什么？（防不住内容篡改）
就像你担心的那样，如果 AI 没有执行 rm，而是执行了类似重定向覆盖或文本替换的命令（例如 sed -i 或者 > original.fq），这个修改指令会穿透软链接，直接篡改底层的原始文件！

3. 终极防御：软链接 + 权限锁死（缺一不可）
为了补全软链接“防不住篡改”的漏洞，我们必须配合文件权限控制。这就是生信圈保护数据的黄金法则（双保险）：

第一道锁：对原始文件剥夺“写权限”（防篡改）
在原始数据存放的根目录，执行：

Bash
chmod -R 444 /path/to/your/raw_data/
这会将所有原始文件变成“只读”（Read-only）。从这一刻起，任何穿透软链接的修改指令都会被 Linux 内核直接拒绝（提示 Permission denied）。

第二道锁：在工作目录建立软链接（防误删）
进入你的工作目录，建立链接：

Bash
ln -s /path/to/your/raw_data/*.fastq.gz .
如果智能体在工作目录里误删文件，它只能删掉快捷方式；如果它试图修改文件内容，会被第一道锁 chmod 444 狠狠弹回。

总结
单独把文件软链接过来，确实不能保护文件免受修改。

安全体系的真相是：用 chmod 444 保护原始文件的内容绝对不被篡改，用 ln -s 把分析现场与原始仓库物理隔离，保护原始文件绝对不被误删。